In [1]:
import time
import random

# ==========================================
# 1. Custom ATM Exception Hierarchy
# ==========================================

class ATMException(Exception):
    """Base exception class for all ATM operations."""
    pass


class InvalidPINException(ATMException):
    """Raised when an incorrect PIN is entered."""
    def __init__(self, remaining_attempts: int):
        self.remaining_attempts = remaining_attempts
        super().__init__(f"Invalid PIN. Remaining attempts: {remaining_attempts}")


class InsufficientFundsException(ATMException):
    """Raised when account balance is less than requested withdrawal amount."""
    def __init__(self, requested: float, balance: float):
        self.requested = requested
        self.balance = balance
        super().__init__(f"Insufficient funds. Requested: ${requested:.2f}, Available: ${balance:.2f}")


class NetworkFailureException(ATMException):
    """Raised when communication with the core banking network fails."""
    def __init__(self, endpoint: str):
        self.endpoint = endpoint
        super().__init__(f"Network connection failed while reaching target endpoint '{endpoint}'.")


# ==========================================
# 2. Bank Account & Core Banking Service
# ==========================================

class BankAccount:
    def __init__(self, pin: str, balance: float):
        self.__pin = pin
        self.balance = balance
        self.pin_attempts = 3

    def verify_pin(self, entered_pin: str):
        if entered_pin != self.__pin:
            self.pin_attempts -= 1
            raise InvalidPINException(remaining_attempts=self.pin_attempts)
        # Reset attempt counter on successful login
        self.pin_attempts = 3


class ATM:
    def __init__(self, account: BankAccount, network_reliability: float = 0.8):
        self.account = account
        self.network_reliability = network_reliability  # Simulated success rate (80%)

    def _simulate_network_request(self):
        """Simulates an upstream RPC network call to the central bank ledger."""
        time.sleep(0.2)  # Simulate network latency
        if random.random() > self.network_reliability:
            raise NetworkFailureException(endpoint="https://bank.internal/api/v1/withdraw")

    def withdraw(self, pin: str, amount: float) -> float:
        """Executes withdrawal sequence with structured error checking."""
        # Step 1: Validate PIN
        self.account.verify_pin(pin)

        # Step 2: Check Account Balance
        if amount > self.account.balance:
            raise InsufficientFundsException(requested=amount, balance=self.account.balance)

        # Step 3: Communicate with Core Banking System over Network
        self._simulate_network_request()

        # Step 4: Deduct funds and complete transaction
        self.account.balance -= amount
        return self.account.balance


# ==========================================
# 3. ATM Transaction Handler (Client Side)
# ==========================================

def process_atm_transaction(atm: ATM, pin: str, amount: float):
    print(f"\n[+] Initiating withdrawal request for ${amount:.2f}...")
    try:
        new_balance = atm.withdraw(pin=pin, amount=amount)
        print(f"[SUCCESS] Dispensing ${amount:.2f}. New Account Balance: ${new_balance:.2f}")

    except InvalidPINException as e:
        print(f"[ERROR - AUTHENTICATION FAILURE] {e}")
        if e.remaining_attempts <= 0:
            print("[SECURITY TRIGGER] Card retained due to multiple incorrect PIN entries.")

    except InsufficientFundsException as e:
        print(f"[ERROR - DECLINED] {e}")
        print(f"Advice: Please select an amount equal to or less than ${e.balance:.2f}.")

    except NetworkFailureException as e:
        print(f"[ERROR - SYSTEM FAILURE] {e}")
        print("Notice: No funds were deducted. Please retry or contact bank support.")

    except ATMException as e:
        print(f"[ERROR - GENERAL ATM EXCEPTION] Unhandled banking error: {e}")

    except Exception as e:
        print(f"[FATAL ERROR] Unexpected system fault: {e}")


# ==========================================
# 4. Verification Test Execution
# ==========================================

if __name__ == "__main__":
    # Create test account: PIN="1234", Balance=$500.00
    user_account = BankAccount(pin="1234", balance=500.00)
    atm_machine = ATM(account=user_account, network_reliability=0.5)

    # Test Case 1: Invalid PIN
    process_atm_transaction(atm_machine, pin="9999", amount=100.00)

    # Test Case 2: Insufficient Funds
    process_atm_transaction(atm_machine, pin="1234", amount=1000.00)

    # Test Case 3: Valid parameters (Network may pass or fail based on 50% simulation)
    process_atm_transaction(atm_machine, pin="1234", amount=200.00)


[+] Initiating withdrawal request for $100.00...
[ERROR - AUTHENTICATION FAILURE] Invalid PIN. Remaining attempts: 2

[+] Initiating withdrawal request for $1000.00...
[ERROR - DECLINED] Insufficient funds. Requested: $1000.00, Available: $500.00
Advice: Please select an amount equal to or less than $500.00.

[+] Initiating withdrawal request for $200.00...
[SUCCESS] Dispensing $200.00. New Account Balance: $300.00
